# SAAM — Part I: Standard Portfolio Allocation
**Region:** North America & Europe (AMER + EUR) &nbsp;|&nbsp; **Scope:** 1 & 2  
**Out-of-sample period:** 2014–2025 &nbsp;|&nbsp; **Rebalancing:** Annual (end of year)

---

This notebook implements the full Part I pipeline:

| Step | Description |
|------|-------------|
| 0 | Setup & imports |
| 1 | Data loading |
| 2 | Region filtering |
| 3 | Data cleaning (prices) |
| 4 | Return computation |
| 5 | Investment set construction |
| 6 | Minimum-variance optimisation |
| 7 | Ex-post portfolio performance |
| 8 | Summary statistics & comparison |
| 9 | Cumulative return plot |
| 10 | Template export |

**Reproducibility:** run all cells top to bottom with no manual intervention.  
Data files must be placed in `data/raw/extracted/` (see `config/paths.example.json`).

## 0. Setup & Imports

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XlImage

# ── Add src/ to path so we can import our utility modules ──────────────────
SRC_DIR = os.path.abspath(os.path.join('..', 'src'))
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from io_utils        import load_ts, clean_prices, compute_returns, ffill_annual, filter_region
from portfolio_utils import (
    get_estimation_window, build_investment_set,
    pairwise_covariance, solve_min_variance,
    compute_mv_weights, compute_mv_returns, compute_vw_returns,
    compute_stats,
)
from plot_utils import plot_cumulative_returns

# ── Paths ──────────────────────────────────────────────────────────────────
CONFIG_PATH = os.path.join('..', 'config', 'paths.json')
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH) as f:
        cfg = json.load(f)
    DATA_DIR    = cfg['data_dir']
    OUTPUT_DIR  = cfg['output_dir']
    FIGURES_DIR = cfg['figures_dir']
    TABLES_DIR  = cfg['tables_dir']
    FILES       = cfg['files']
else:
    # Default fallback (adjust if needed)
    DATA_DIR    = os.path.join('..', 'data', 'raw', 'extracted') + os.sep
    OUTPUT_DIR  = os.path.join('..', 'output') + os.sep
    FIGURES_DIR = os.path.join('..', 'results', 'figures') + os.sep
    TABLES_DIR  = os.path.join('..', 'results', 'tables') + os.sep
    FILES = {
        'static':   'Static_2025.xlsx',
        'ri_m':     'DS_RI_T_USD_M_2025.xlsx',
        'mv_m':     'DS_MV_T_USD_M_2025.xlsx',
        'mv_y':     'DS_MV_T_USD_Y_2025.xlsx',
        'co2_s1':   'DS_CO2_SCOPE_1_Y_2025.xlsx',
        'co2_s2':   'DS_CO2_SCOPE_2_Y_2025.xlsx',
        'rev':      'DS_REV_Y_2025.xlsx',
        'rf':       'Risk_Free_Rate_2025.xlsx',
        'template': 'Template_for_Part_I-SAAM.xlsx',
    }

for d in [OUTPUT_DIR, FIGURES_DIR, TABLES_DIR]:
    os.makedirs(d, exist_ok=True)

print('✓ Libraries loaded')
print(f'  Data dir    : {DATA_DIR}')
print(f'  Figures dir : {FIGURES_DIR}')

## 1. Data Loading

Each Datastream export has:
- **Row 0:** error messages (`$$ER: E100,...`) → skipped automatically by `load_ts()`
- **Column `ISIN`:** firm identifier → used as index
- **Remaining columns:** date-indexed time series

In [ ]:
static = pd.read_excel(DATA_DIR + FILES['static'])

ri_m   = load_ts(DATA_DIR + FILES['ri_m'])
mv_m   = load_ts(DATA_DIR + FILES['mv_m'])
mv_y   = load_ts(DATA_DIR + FILES['mv_y'])
co2_s1 = load_ts(DATA_DIR + FILES['co2_s1'])
co2_s2 = load_ts(DATA_DIR + FILES['co2_s2'])
rev    = load_ts(DATA_DIR + FILES['rev'])

rf_df = pd.read_excel(DATA_DIR + FILES['rf'])
rf_df.columns = ['Date', 'RF']

print(f"Static:        {static.shape[0]} firms")
print(f"RI monthly:    {ri_m.shape}")
print(f"MV monthly:    {mv_m.shape}")
print(f"MV annual:     {mv_y.shape}")
print(f"CO2 Scope 1:   {co2_s1.shape}")
print(f"CO2 Scope 2:   {co2_s2.shape}")
print(f"Revenue:       {rev.shape}")
print(f"\nRegion breakdown: {dict(static['Region'].value_counts())}")

## 2. Region Filtering — North America & Europe

We keep firms assigned to `AMER` (North America) and `EUR` (Europe) as per our group's strategy.

In [ ]:
REGIONS = ['AMER', 'EUR']

dfs = filter_region(
    {'ri_m': ri_m, 'mv_m': mv_m, 'mv_y': mv_y,
     'co2_s1': co2_s1, 'co2_s2': co2_s2, 'rev': rev},
    static, REGIONS
)
ri_m, mv_m, mv_y = dfs['ri_m'], dfs['mv_m'], dfs['mv_y']
co2_s1, co2_s2, rev = dfs['co2_s1'], dfs['co2_s2'], dfs['rev']

# Sort price columns chronologically
date_cols = sorted([c for c in ri_m.columns if hasattr(c, 'year')])
ri_m = ri_m[date_cols]
mv_m = mv_m[[c for c in date_cols if c in mv_m.columns]]

print(f"Firms in {REGIONS}: {ri_m.shape[0]}")
print(f"Date range: {date_cols[0].strftime('%Y-%m')} → {date_cols[-1].strftime('%Y-%m')}")

## 3. Data Cleaning — Prices

Per project instructions:

1. **Low prices** (`RI < 0.5`) → `NaN` to avoid extreme/infinite returns.
2. **Intermediate forward-fill** — gaps between the first and last valid price
   correspond to misreporting and are filled with the previous value.
3. Prices at the **start** (firm not yet listed) and at the **end** (after delisting)
   remain `NaN`.

In [ ]:
n_low_before = (ri_m < 0.5).sum().sum()
ri_m_clean = clean_prices(ri_m, low_price_threshold=0.5)
n_low_after = ri_m_clean.isna().sum().sum() - ri_m.isna().sum().sum()

print(f"Prices < 0.5 set to NaN : {n_low_before:,}")
print(f"Price matrix shape      : {ri_m_clean.shape}")

## 4. Return Computation

Simple monthly returns: $R_{i,t} = P_{i,t} / P_{i,t-1} - 1$

| Price at $t-1$ | Price at $t$ | Return |
|---|---|---|
| `NaN` | any | `NaN` (not yet listed) |
| valid | `NaN` (permanent) | $-1$ (delisting) |
| valid | valid | $P_t / P_{t-1} - 1$ |

In [ ]:
returns   = compute_returns(ri_m_clean)
ret_dates = list(returns.columns)

# Sanity check: no intermediate NaN should remain
n_intermediate = 0
for isin in returns.index:
    r  = returns.loc[isin].values
    fv = next((j for j in range(len(r)) if not np.isnan(r[j])), None)
    lv = next((j for j in range(len(r)-1, -1, -1) if not np.isnan(r[j])), None)
    if fv is not None and lv is not None:
        n_intermediate += int(np.isnan(r[fv:lv+1]).sum())

n_delistings = int((returns == -1.0).sum().sum())

print(f"Returns matrix          : {returns.shape[0]} firms × {returns.shape[1]} months")
print(f"Date range              : {ret_dates[0].strftime('%Y-%m')} → {ret_dates[-1].strftime('%Y-%m')}")
print(f"Intermediate NaN        : {n_intermediate} (should be 0)")
print(f"Delisting events (−100%): {n_delistings}")

## 5. Annual Data Forward-Fill

CO₂ and revenue data are annual. Missing values between two valid observations
or at the end of the sample are forward-filled.  Missing values at the **start**
remain `NaN` (firm excluded from investment set until data becomes available).

In [ ]:
co2_s1_f = ffill_annual(co2_s1)
co2_s2_f = ffill_annual(co2_s2)
rev_f    = ffill_annual(rev)

print('✓ Forward-fill complete for CO₂ Scope 1, Scope 2, and Revenue')

## 6. Investment Set Construction

For each year $Y \in \{2013, \ldots, 2024\}$, a firm is eligible if:

1. Valid price at end of $Y$
2. ≥ 36 valid monthly returns in the 120-month window
3. Proportion of zero returns ≤ 50% (stale-price filter)
4. Scope 1, Scope 2, and Revenue data available for year $Y$

> Applying the carbon filter from Part I ensures a **consistent investment set**
> across both parts of the project.

In [ ]:
investment_sets = {}

print(f"{'Year':<6s} {'Eligible':>8s}")
print("-" * 16)

for Y in range(2013, 2025):
    iset = build_investment_set(
        Y, returns, ri_m_clean, co2_s1_f, co2_s2_f, rev_f, ret_dates,
        min_obs=36, stale_threshold=0.50,
    )
    investment_sets[Y] = iset
    print(f"{Y:<6d} {len(iset):>8d}")

## 7. Minimum-Variance Portfolio Optimisation

For each year $Y$, we solve:

$$
\min_{\alpha_Y} \; \alpha_Y' \Sigma_Y \alpha_Y
\quad \text{s.t.} \quad
\alpha_Y' \mathbf{e} = 1, \quad \alpha_{i,Y} \geq 0 \; \forall i
$$

$\Sigma_Y$ is estimated via **pairwise complete observations** to avoid discarding
firms with NaN only at the beginning of the estimation window.

In [ ]:
mv_weights = compute_mv_weights(investment_sets, returns, ret_dates)

### 7.1 Top Holdings

In [ ]:
static_idx = static.set_index('ISIN')

for Y in [2013, 2017, 2021, 2024]:
    w    = mv_weights[Y]
    top10 = w.nlargest(10)
    print(f"\nYear {Y} — Top 10 holdings")
    print(f"  {'ISIN':<14s} {'Name':<40s} {'Weight':>7s}")
    print("  " + "-" * 63)
    for isin, wt in top10.items():
        name = static_idx.loc[isin, 'NAME'] if isin in static_idx.index else 'Unknown'
        print(f"  {isin:<14s} {str(name)[:40]:<40s} {wt*100:>6.2f}%")

## 8. Ex-Post Portfolio Returns

### 8.1 Minimum-Variance

Weights are set at end of $Y$ and drift each month:

$$
\alpha_{i,t+k} = \frac{\alpha_{i,t+k-1}(1+R_{i,t+k})}{1+R_{p,t+k}}
$$

### 8.2 Value-Weighted (Benchmark)

$$
R^{(vw)}_{t+1} = \sum_i w_{i,t} R_{i,t+1}, \quad w_{i,t} = \frac{\text{Cap}_{i,t}}{\sum_j \text{Cap}_{j,t}}
$$

In [ ]:
mv_ret = compute_mv_returns(investment_sets, mv_weights, returns, ret_dates)
vw_ret = compute_vw_returns(investment_sets, returns, mv_m, ret_dates)

# Align on common dates
common = mv_ret.index.intersection(vw_ret.index)
mv_ret = mv_ret.loc[common]
vw_ret = vw_ret.loc[common]

print(f"MV portfolio: {len(mv_ret)} monthly returns  "
      f"({mv_ret.index[0].strftime('%Y-%m')} → {mv_ret.index[-1].strftime('%Y-%m')})")
print(f"VW portfolio: {len(vw_ret)} monthly returns  "
      f"({vw_ret.index[0].strftime('%Y-%m')} → {vw_ret.index[-1].strftime('%Y-%m')})")

## 9. Summary Statistics

In [ ]:
# Align risk-free rate
rf_df['year']  = rf_df['Date'].astype(int) // 100
rf_df['month'] = rf_df['Date'].astype(int) % 100
rf_map = {}
for _, row in rf_df.iterrows():
    for d in common:
        if d.year == int(row['year']) and d.month == int(row['month']):
            rf_map[d] = row['RF'] / 100
            break
rf = pd.Series(rf_map).reindex(common).fillna(0)

vw_stats = compute_stats(vw_ret, rf)
mv_stats = compute_stats(mv_ret, rf)

stats_df = pd.DataFrame({'Value-Weighted': vw_stats, 'Min-Variance': mv_stats})
print(stats_df.applymap(lambda x: f"{x:.4f}").to_string())

# Save to CSV
stats_df.to_csv(TABLES_DIR + 'summary_stats_part1.csv')
print(f"\n✓ Saved to {TABLES_DIR}summary_stats_part1.csv")

## 10. Cumulative Return Plot

In [ ]:
fig = plot_cumulative_returns(
    series_dict = {'VW': vw_ret, 'MV': mv_ret},
    title = 'Cumulative Performance: VW vs. Minimum-Variance Portfolio\n(North America & Europe, 2014–2025)',
    save_path = FIGURES_DIR + 'cumulative_returns_part1.png',
)
plt.show()
print(f"\nFinal growth — VW: ${(1+vw_ret).prod():.4f} | MV: ${(1+mv_ret).prod():.4f}")

## 11. Template Export

In [ ]:
wb = load_workbook(DATA_DIR + FILES['template'])
ws = wb.active

stat_keys = [
    'Annualized average return',
    'Annualized volatility',
    'Annualized cumulative return',
    'Sharpe ratio',
    'Minimum monthly return',
    'Maximum monthly return',
]
for i, key in enumerate(stat_keys):
    ws.cell(row=i + 3, column=2, value=vw_stats[key])
    ws.cell(row=i + 3, column=3, value=mv_stats[key])

for i in range(len(mv_ret)):
    ws.cell(row=i + 3, column=6, value=float(vw_ret.iloc[i]))
    ws.cell(row=i + 3, column=7, value=float(mv_ret.iloc[i]))

try:
    img = XlImage(FIGURES_DIR + 'cumulative_returns_part1.png')
    img.width, img.height = 600, 300
    ws.add_image(img, 'B10')
except Exception as e:
    print(f'Note: could not insert image — {e}')

out_path = OUTPUT_DIR + 'Template_Part_I_FILLED.xlsx'
wb.save(out_path)
print(f'✓ Template saved → {out_path}')
print('\n✓ Part I complete.')